# Vehicle Price Monitor

## AUTO: **Fiat Panda**

**Vehicle Price Monitor** è un’applicazione Python progettata per raccogliere e analizzare annunci di auto usate presenti su AutoScout24 o altri siti specializzati, tramite tecniche di web scraping (ad esempio con *BeautifulSoup*, o software come Octoparse).

🎯 **Obiettivo**: aiutare l’utente a individuare le migliori offerte di veicoli in base a criteri oggettivi e personalizzabili, come prezzo, chilometraggio, potenza, distanza e allestimento.


#### 🔍 Funzionalità principali:
- Calcolo della **distanza** e analisi del **prezzo**, **chilometraggio**, **potenza** e **allestimento**.
- Calcolo di un **Indice di Appetibilità** personalizzato.
- Restituzione delle **10 migliori occasioni** con indicazione se ciascuna rappresenta un buon affare rispetto al mercato..

#### 🧠 Indice di Appetibilità
L'indice misura l'interesse di un'auto considerando: Anno, prezzo, chilometraggio, distanza, allestimento, potenza e cambio automatico.

#### 🛠️ Come usare:
1. Carica un dataset con i dati delle auto o l'**URL di AutoScout24** per lo scraping.
2. Aggiungi il tuo **comune di residenza**.
3. Personalizza l'**Indice di Appetibilità** con i tuoi pesi preferiti

# 0. Importazioni

## 0.1. Librerie

In [ ]:
# Standard Library
import os
import csv
import time
import re
import random
import subprocess
import tempfile
import unicodedata
from datetime import datetime

# Web Scraping / HTTP
import requests
from bs4 import BeautifulSoup

# Geolocation
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

# Data Analysis
import numpy as np
import pandas as pd
from scipy.stats import norm
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.tsaplots import plot_acf

# Machine Learning
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score, classification_report

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Jupyter Display
from IPython.display import display, HTML

In [ ]:
#per salvare i plot plotly in pdf:
#!pip install kaleido -q
#import kaleido #required
#kaleido.__version__ #0.2.1
#import plotly
#plotly.__version__ #5.5.0

In [ ]:
raw = 'https://raw.githubusercontent.com/'

## 0.2. Scraping Autoscout

Lo scraper di AutoScout estrae circa 400 risultati per volta, poiché il sito mostra un massimo di 400 annunci alla volta.
Per ottenere un dataset più completo, è consigliabile eseguire lo scraping più volte, suddividendo le ricerche in base a criteri come le fasce di chilometraggio. In questo modo è possibile aggirare il limite e raccogliere un numero maggiore di annunci.

In [ ]:
def download_scraper_if_needed():
    url = "https://raw.githubusercontent.com/FedeGambe/Vehicle_Price_Monitor/main/autoscout_scraper.py"
    filename = "autoscout_scraper.py"

    if not os.path.exists(filename):
        print("Scarico lo scraper da GitHub...")
        try:
            subprocess.run(["wget", url], check=True)
            print("Scraper scaricato con successo!")
        except FileNotFoundError:
            print("wget non trovato, provo con curl...")
            subprocess.run(["curl", "-O", url], check=True)
    else:
        print("autoscout_scraper.py già presente.")

download_scraper_if_needed()

# Ora puoi importare la funzione
from autoscout_scraper import autoscout_scraper
print("Scraper importato con successo!")

In [ ]:
##############################
# Per utilizzare lo scraper: #
##############################

#url_autoscout = "https://www.autoscout24.it/lst/toyota/yaris?atype=C&cq_cmp=22176097133&cq_net=g&cq_plt=gp&cq_src=google_ads&cy=I&damaged_listing=exclude&desc=0&fregfrom=2019&gad_source=1&gclid=Cj0KCQjw782_BhDjARIsABTv_JB66fHjATvYzqM6MgWXdMD8S2IvCeLbO5kwI0RLPnUHEUU_rqaGvCUaAkC_EALw_wcB&lat=44.66517&lon=11.135&powertype=kw&search_id=28ja0n5dc5n&sort=standard&ustate=N%2CU&utm_campaign=IT_AS24_ugom_web_lead_%5Bs%5D_%28DSA%29_DSA&utm_content=%28CM%29_Audi_DSA&utm_medium=cpc&utm_source=google&zip=40019-sant%27agata-bolognese&zipr=100"
#autoscout_scraper(url_autoscout)

## 0.3. Funzioni

In [ ]:
url_funzioni = "https://raw.githubusercontent.com/FedeGambe/Vehicle_Price_Monitor/main/utils/funzioni/"

def download_funzioni(filename):
    url = url_funzioni + filename

    if not os.path.exists(filename):
        print(f"Scarico le funzioni di {filename} da GitHub...")
        try:
            subprocess.run(["wget", url], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"Le funzioni di {filename} scaricate con successo!")
        except FileNotFoundError:
            print("wget non trovato, provo con curl...")
            try:
                subprocess.run(["curl", "-O", url], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                print(f"Le funzioni di {filename} scaricate con successo!")
            except subprocess.CalledProcessError:
                print(f"Errore durante il download di {filename} con curl.")
    else:
        print(f"{filename} già presente.")

In [ ]:
download_funzioni("pulizia.py")
download_funzioni("norm_testo.py")

In [ ]:
from pulizia import pulisci_prezzo, pulisci_km, pulisci_cambio, pulisci_cambio_dt_merged, pulisci_carburante, pulisci_indirizzo_AT
print("- Pulizia importato con successo!")

from norm_testo import normalizza_testo
print("- Normalizza Testo importato con successo!")

# 1. Import datasets

## 1.1. Località

### 1.1.1. Comuni, CAP e regioni

In [ ]:
url_cap = raw + 'FedeGambe/Vehicle_Price_Monitor/main/Materiali/Data/gi_comuni_cap.csv'
cap= pd.read_csv(url_cap, delimiter=';')

only_cap = cap[['cap', 'denominazione_ita', 'sigla_provincia','denominazione_regione', 'ripartizione_geografica']]
only_cap = only_cap.rename(columns={'denominazione_ita': 'Comune', 'cap': 'CAP', 'sigla_provincia': 'Provincia', 'ripartizione_geografica': 'Area', 'denominazione_regione': 'Regione'})

In [ ]:
only_cap['Comune'] = only_cap['Comune'].apply(normalizza_testo)
only_cap['Provincia'] = only_cap['Provincia'].apply(normalizza_testo)

print("Cap di tutta italia",only_cap.info(), "\n")

only_cap_per_comune = only_cap.sort_values('CAP').drop_duplicates(subset='Comune', keep='first')
only_comune_per_cap = only_cap.sort_values('Comune').drop_duplicates(subset='CAP', keep='first')
print("Cap duplicati eliminati delle grandi città\n",only_comune_per_cap.info(), "\n")
print("Comune duplicati eliminati dai cap\n",only_cap_per_comune.info())

### 1.1.2. Latitutide e Longitudine

In [ ]:
url_distanza = raw + 'FedeGambe/Vehicle_Price_Monitor/main/Materiali/Data/italy_geo.json'
distanza = pd.read_json(url_distanza)
distanza = distanza.rename(columns={'comune': 'Comune', 'lng': 'Longitudine', 'lat': 'Latitudine'})
distanza.info()

In [ ]:
distanza['Comune'] = distanza['Comune'].apply(normalizza_testo)
distanza.drop(columns=['istat'], inplace=True)
distanza = distanza.drop([7978, 7979])
distanza.head(5)

In [ ]:
# Carica GeoJSON delle regioni italiane
url_reg_json = 'https://raw.githubusercontent.com/openpolis/geojson-italy/master/geojson/limits_IT_regions.geojson'
geojson = requests.get(url_reg_json).json()

## 1.2. Dataset Automobile.it

In [ ]:
#data_AT = pd.read_csv('/content/Scraper Automobile.it - panda.csv', sep=',')
data_AT = pd.read_csv('https://raw.githubusercontent.com/FedeGambe/Vehicle_Price_Monitor/main/Esempi/Datasets/Scraper%20Automobile.it%20-%20panda.csv', sep=',')
data_AT.info()

# 2. Preparazione Dataset x Analisi

## 2.1. Pulizia Dataset

In [ ]:
data_AT['Prezzo'] = data_AT['Prezzo'].apply(pulisci_prezzo).astype('Int64')
data_AT.loc[data_AT['Chilometraggio'].str.contains('CV', na=False), 'Chilometraggio'] = 0
data_AT['Chilometraggio'] = data_AT['Chilometraggio'].apply(pulisci_km).astype('Int64')
data_AT ['Immatricolazione'] = data_AT['Immatricolazione'].str.extract(r'(\d{4})').astype('Int64')

In [ ]:
data_AT[data_AT['Chilometraggio'] > 300000]

In [ ]:
data_AT = data_AT[data_AT['Chilometraggio'] <= 300000].copy()

In [ ]:
valori_cambio = ['Manuale', 'Automatico', 'Semiautomatico'] # Lista dei valori che dovrebbero stare nella colonna 'Cambio'
mask_cambio = data_AT['Carburante'].isin(valori_cambio) # Crea una maschera per le righe che hanno questi valori in 'Carburante'
data_AT.loc[mask_cambio, 'Cambio'] = data_AT.loc[mask_cambio, 'Carburante'] # Sposta i valori nella colonna 'Cambio'
data_AT.loc[mask_cambio, 'Carburante'] = pd.NA # Svuota la colonna 'Carburante' in quelle righe

In [ ]:
data_AT['Cambio'] = data_AT.apply(lambda row: pulisci_cambio(row['Cambio'], row['Annuncio']), axis=1)

In [ ]:
print(data_AT['Carburante'].unique())

In [ ]:
# Carburanti riconosciuti
carburanti_validi = ['Diesel', 'Benzina', 'Ibrido', 'Metano', 'GPL']
def pulisci_carburante(valore, valore2):
    def normalize(val):
        return '' if pd.isna(val) else str(val).strip().lower()
    carburante_raw = normalize(valore)
    annuncio = normalize(valore2)
    for c in carburanti_validi:
        if c.lower() in carburante_raw:
            return c
    if any(k in annuncio for k in ['mild', 'plug', 'mhv']):
        return 'Ibrido'
    if 'Diesel' in carburante_raw:
        return 'Diesel'
    if 'Ibrido' in carburante_raw or 'Hybrid' in carburante_raw or 'elettric' in carburante_raw:
        return 'Ibrido'
    if 'Metano' in carburante_raw:
        return 'Metano'
    if 'GPL' in carburante_raw:
        return 'GPL'
    return 'Benzina'

In [ ]:
data_AT['Carburante'] = data_AT.apply(lambda row: pulisci_carburante(row['Carburante'], row['Annuncio']), axis=1)
print(data_AT['Carburante'].unique())

In [ ]:
# Mappatura modelli e motorizzazioni per Fiat Panda 2012-in corso
motorizzazioni_panda = {
    "1.2": "1.2 Fire", "panda12": "1.2 Fire", "1.269cv": "1.2 Fire",
    "0.9": "0.9 TwinAir", "twinair": "0.9 TwinAir", "panda09": "0.9 TwinAir", "natural power" : "0.9 TwinAir", "natural" : "0.9 TwinAir",
    "1.0": "1.0 Hybrid", "hybrid": "1.0 Hybrid", "panda10": "1.0 Hybrid",
    "1.3": "1.3 MJT", "mjt": "1.3 MJT", "multijet": "1.3 MJT",
}
modelli_panda_ord = sorted(motorizzazioni_panda.keys(), key=len, reverse=True)

# Cavalli per modello
mappa_cv_panda = {
    "1.2 Fire": 69,
    "0.9 TwinAir": 85,
    "1.0 Hybrid": 70,
    "1.3 MJT": 95
}

def estrai_modello(annuncio):
    if not isinstance(annuncio, str):
        return None
    annuncio_norm = re.sub(r"[^\w]", "", annuncio.lower())
    for chiave in modelli_panda_ord:
        if chiave in annuncio_norm:
            return motorizzazioni_panda[chiave]
    return None

def estrai_cv(modello):
    return mappa_cv_panda.get(modello)

In [ ]:
#data_AT['Modello'] = data_AT['Annuncio'].apply(estrai_modello)
data_AT['Modello'] = data_AT.apply(lambda row: estrai_modello(row['Annuncio']) or estrai_modello(row['Link']),axis=1)
data_AT['CV'] = data_AT['Modello'].apply(estrai_cv)

In [ ]:
data_AT = data_AT.rename(columns={'TIpologia': 'Tipologia'})
data_AT = data_AT.drop(columns=['Nome_Venditore', 'Unico_Proprietario', 'Tipologia'])

In [ ]:
condizione = (
    data_AT['Immatricolazione'].isnull() &
    (data_AT['Prezzo'] > 12000) &
    data_AT['Chilometraggio'].isnull() | (data_AT['Chilometraggio'] < 1000)
)

data_AT[condizione]

In [ ]:
data_AT.loc[condizione, 'Immatricolazione'] = 2025

In [ ]:
data_AT[['Comune', 'Provincia']] = data_AT['Località'].str.extract(r'(.+)\s+\((\w{2})\)')
data_AT['Comune'] = data_AT['Comune'].apply(normalizza_testo)
data_AT = data_AT.drop(columns=['Località', 'Provincia'])
data_AT = data_AT.merge(only_cap_per_comune[['CAP', 'Comune']], on='Comune', how='left')

## 2.2. Preparazione dataset finale con località

In [ ]:
data = data_AT.copy()
data.info()

In [ ]:
data = data.merge(only_cap_per_comune[['Regione', 'Comune']], on='Comune', how='left')
data = data.merge(only_cap_per_comune[['Area', 'Comune']], on='Comune', how='left')

In [ ]:
data.info()

In [ ]:
val_doppi = data['Link'].duplicated()
print(f"Ci sono dei valori doppi? {val_doppi.any()}")
if val_doppi.any() == True:
    print(f"Se si quanti ce ne sono? {val_doppi.sum()}, su un totale di {len(data)}")
    data = data.drop_duplicates(subset='Link', keep='first')
    print("Valori doppi rimossi")
else:
    print("Non ci sono valori doppi")

## 2.3. Calcolo Distanza

In [ ]:
distanza

In [ ]:
#comune_riferimento = str(input("Inserisci il tuo comune di residenza")) # Filtra il comune di riferimento
comune_riferimento = "Sant'Agata Bolognese" # Filtra il comune di riferimento
comune_riferimento = normalizza_testo(comune_riferimento)
comune_rif = distanza[distanza['Comune'] == comune_riferimento]

# Estrazione latitudine e longitudine per il comune di riferimento
lat_rif = comune_rif['Latitudine'].values[0]
lon_rif = comune_rif['Longitudine'].values[0]

# Funzione per calcolare la distanza tra il comune di riferimento e gli altri comuni
def calcola_distanza(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

# Calcolare la distanza per tutti gli altri comuni
distanza['Distanza'] = distanza.apply(
    lambda row: calcola_distanza(
        lat_rif,
        lon_rif,
        row['Latitudine'],
        row['Longitudine'])
    , axis=1)

distanza_sorted = distanza.sort_values(by='Distanza')
distanza_sorted[['Comune', 'Distanza']].head(10)

In [ ]:
only_cap_per_comune

In [ ]:
distanza_sorted_provi = distanza_sorted.merge(only_cap_per_comune[['Comune', 'Provincia']], on='Comune', how='left')
distanza_sorted_provi = distanza_sorted_provi.sort_values(by='Distanza')
distanza_sorted_provi

Unione distanze con il Dataframe `Data`

In [ ]:
data = data.merge(distanza[['Comune', 'Distanza']], on='Comune', how='left')
data['Distanza'] = data['Distanza'].round(2)
data.info()

In [ ]:
print("Null value di Distanza:",data['Distanza'].isna().sum())
print("Null value di Città:",data['Comune'].isna().sum())
print("Null value di CAP:",data['CAP'].isna().sum())

## 2.4. Formattazione

In [ ]:
if 'Località' in data.columns:
    data = data.drop(columns=['Località'])

#data['Venditore'] = data['Venditore'].fillna('Rivenditore') #riempire i valori vuoti con Rivenditori, in quanto non sono provati
data['Cambio'] = data.apply(lambda row: pulisci_cambio_dt_merged(row['Cambio'], row['Annuncio'], row['Link']), axis=1) #nel caso non rilevi dei cambi
data ['Anni'] = data.apply(lambda row: estrai_anni(row['Immatricolazione'], row['Chilometraggio'], row['Prezzo']), axis=1) #calcolo Anni

data = data.dropna(subset=['Annuncio']) #Per ricavare l'allestimento

data = data.dropna()

for col in ['Prezzo', 'Immatricolazione', 'Chilometraggio', 'CV', 'Distanza', 'CAP', 'Anni']:
    data[col] = data[col].astype(int)

#Nuovo Ordine
data = data[['Annuncio', 'Link', 'Prezzo', 'Anni' ,'Immatricolazione', 'Chilometraggio', 'Cambio',
             'Carburante', 'CV','Modello', 'Distanza','Comune', 'CAP', 'Regione', 'Area']]

data.info()

In [ ]:
duplicati = data['Link'].duplicated().sum()
print(f"Numero di righe duplicate: {duplicati}")
data_data_red = data.drop_duplicates(subset='Link')

n_unici = data['Link'].nunique()
print(f"Numero di link unici: {n_unici}")

## 2.5. Dataset dummy

In [ ]:
data_dummy = data.copy()
data_dummy= data_dummy.drop (columns=['Annuncio', 'Link', 'Comune','Immatricolazione', 'CAP', 'Modello', 'Regione'])

In [ ]:
data_dummy.select_dtypes(include=['object']).info()

In [ ]:
data_dummy['Cambio'] = data_dummy['Cambio'].map({'automatico': 1, 'manuale': 0})
#data_dummy['Venditore'] = data_dummy['Venditore'].map({'Privato': 0, 'Rivenditore': 1})

In [ ]:
print('I carburanti presenti sono:', pd.unique(data['Carburante']))

carburanti = pd.unique(data_dummy['Carburante'])
for carburante in carburanti:
    if carburante != 'Benzina': # Benzina = baseline
        data_dummy[f'is_{carburante} vs Benzina'] = data_dummy['Carburante'].apply(lambda x: 1 if x == carburante else 0)
data_dummy.drop(columns=['Carburante'], inplace=True)

In [ ]:
print(pd.unique(data_dummy['Area']))

In [ ]:
nord_e = ['Nord-est'] # baseline
nord_o = ['Nord-ovest']
centro = ['Centro']
sud_isole = ['Sud', 'Isole']

data_dummy['is NO vs NE']= data_dummy['Area'].isin(nord_o).astype(int)
data_dummy['is Centro vs NE']= data_dummy['Area'].isin(centro).astype(int)
data_dummy['is Sud_Isole vs NE']= data_dummy['Area'].isin(sud_isole).astype(int)
data_dummy.drop(columns=['Area'], inplace=True)

In [ ]:
data_dummy.info()

# 3. Analisi
Si cerca di comprendere il prezzo (par. 3.1.) per stabilire se il veicolo analizzato è conveniente o meno, se è 1: conveniente mentre se è0: non conveniente (par. 3.2.).
Infine, si calcola un indice di appetibilità, il quale prende in considerazione diverse caratteristiche dell’automobile per individuare la migliore (par. 4.3.).

## 3.1. Comprendere il prezzo

In [ ]:
X = data_dummy.drop(columns=['Prezzo', 'Distanza'])
y = data_dummy['Prezzo']

In [ ]:
#@title Correlazione delle variabili indipendenti con il prezzo
corr_matrix = data_dummy.corr()
target_corr = corr_matrix['Prezzo'].drop('Prezzo')
target_corr_sorted = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)

corr_df = pd.DataFrame({
    'Variabile': target_corr_sorted.index,
    'Correlazione': target_corr_sorted.values})


fig_cor_y = px.bar(corr_df,x='Correlazione',y='Variabile',
    orientation='h',
    color='Correlazione', color_continuous_scale='RdBu')

fig_cor_y.update_traces(hovertemplate='<b>%{y}</b><br>Correlazione: %{x:.2f}')
fig_cor_y.update_layout(title={'text': "Correlazione delle feature con il Prezzo",'x': 0.5,
                         'xanchor': 'center', 'font': {'size': 20, 'family': 'Arial', 'weight': 'bold'},},
                  yaxis=dict(autorange="reversed"),  height=600, width=800, )
fig_cor_y.show()

In [ ]:
#@title Correlazione tra le variabili indipendenti
correlation_matrix = X.corr()
original_columns = list(correlation_matrix.columns)

fig_corr_X = ff.create_annotated_heatmap(z=correlation_matrix.values, x=original_columns, y=original_columns,
    annotation_text=np.around(correlation_matrix.values, decimals=2),showscale=True, colorscale='RdBu',reversescale=True,hoverinfo='text',)

fig_corr_X.update_layout(
    title={'text': "Correlation Matrix tra le features",'x': 0.5,'xanchor': 'center',  'font': {'size': 20, 'family': 'Arial', 'weight': 'bold'},},
    margin=dict(t=200, b=25, l=0, r=0),height=600,width=1000)
fig_corr_X.show()

In [ ]:
X = X.astype(float)
y = y.astype(float)
X_cons = sm.add_constant(X)

In [ ]:
vif_data = pd.DataFrame()
vif_data['Variable'] = X_cons.columns
vif_data['VIF'] = [variance_inflation_factor(X_cons.values, i) for i in range(X_cons.shape[1])]
vif_data

In [ ]:
model = sm.OLS(y, X_cons)
results = model.fit()
print(results.summary())
residui = results.resid
plot_acf(residui)
plt.title('Autocorrelazione dei residui')
plt.show()

In [ ]:
coefficienti_b = round(results.params,3)
p_value = round(results.pvalues,3)
standard_errors =round(results.bse,3)

tabella_diz = {
    'Odds Ratio': np.exp(coefficienti_b),
    'Coefficienti': coefficienti_b,
    'P-value': p_value,
    'Standard Error': standard_errors
}
tabella_riass = pd.DataFrame(tabella_diz)
tabella_riass.index = results.params.index
tabella_sign = tabella_riass[tabella_riass['P-value'] < 0.05]

tabella_sign['indice'] = range(len(tabella_sign))
tabella_sign

In [ ]:
X = X.drop(columns=['Cambio']) #per valori di VIF alto e
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
reg = linear_model.LinearRegression()
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
print(f"Mean Squared Error: {round(mean_squared_error(y_test, y_pred),2)}")
print(f"Root Mean Squared Error: {round(np.sqrt(mean_squared_error(y_test, y_pred)),2)}")
print(f"R-squared: {round(r2_score(y_test, y_pred),2)}")

In [ ]:
#print('Cambio:', data[['Cambio']].value_counts(normalize=True), '\n')
print('Carburante:', data[['Carburante']].value_counts(normalize=True), '\n')
print('Area:' , data[['Area']].value_counts(normalize=True))

In [ ]:
#@title Confronto Variabili: Regressione Polinomiale
grado = 3
titoli = list(X.columns)
# Calculate the number of rows and columns needed for subplots
n_cols = 3  # Number of columns
n_rows = (len(titoli) + n_cols - 1) // n_cols  # Calculate rows needed

fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titoli)

for i in range(len(titoli)): # Loop through the number of features
    x = X.iloc[:, i].values.reshape(-1, 1)
    # Convert x to a NumPy array before calling flatten()
    x = np.array(x)
    y_vals = y.values

    # -------------------------------
    # 🔸 Regressione Polinomiale
    grado = 3
    poly = PolynomialFeatures(degree=grado)
    x_poly = poly.fit_transform(x)
    model = LinearRegression().fit(x_poly, y_vals)

    x_range = np.linspace(x.min(), x.max(), 300).reshape(-1, 1)
    x_range_poly = poly.transform(x_range)
    y_pred_poly = model.predict(x_range_poly)

    # -------------------------------
    # Tracce
    row = i // n_cols + 1 # Calculate the correct row for the subplot
    col = i % n_cols + 1  # Calculate the correct column for the subplot

    # Dati ,marker=dict( opacity=0.6)
    fig.add_trace(go.Scatter(x=x.flatten(), y=y_vals,mode='markers', marker=dict(size=3, color='deepskyblue', opacity=0.3),name='Dati',showlegend = False),row=row, col=col) #per legenda mettere: showlegend=(i == 0)
    # Polinomiale
    fig.add_trace(go.Scatter(x=x_range.flatten(), y=y_pred_poly,mode='lines',line=dict(color='orange', width=2),name=f'Polinomiale di grado: {grado}',showlegend = False),row=row, col=col) #per legenda mettere: showlegend=(i == 0)

    fig.update_yaxes(title_text="Prezzo", row=row, col=col, title_font=dict(size=10), tickfont=dict(size=9))

fig.update_layout(
    title={'text': f"Confronto: Regressione Polinomiale (grado {grado})", 'x': 0.5, 'xanchor': 'center', 'font': {'size': 20, 'family': 'Arial', 'weight': 'bold'}},
    template='plotly_dark',height=1200,width=1500, )

fig.update_annotations(font_size=10)
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()

In [ ]:
data_filt_reg = data[(data['Chilometraggio'] <= 40000)]

# suddivisione per carburante
carburante1 = data_filt_reg[data_filt_reg['Carburante'] =='Diesel']
carburante2 = data_filt_reg[data_filt_reg['Carburante'] =='Benzina']
carburante3 = data_filt_reg[data_filt_reg['Carburante'] =='Ibrido']
carburante4 = data_filt_reg[data_filt_reg['Carburante'] =='GPL']
carburante5 = data_filt_reg[data_filt_reg['Carburante'] =='Metano']

# funzione per calcolo media per regione
def get_map_trace(df, geojson, title, color_scale):
    if df.empty:
        return go.Choropleth(locations=[], z=[], geojson=geojson) #se non c'è lauto in una det. regione verrà resituto un valore vuoto
    media = df.groupby("Regione")["Prezzo"].mean().reset_index()
    return go.Choropleth(geojson=geojson,featureidkey="properties.reg_name",locations=media["Regione"],
        z=media["Prezzo"],colorscale=color_scale,colorbar_title="Prezzo (€)",
        zmin=data_filt_reg['Prezzo'].min(),zmax=data_filt_reg['Prezzo'].max(),hovertemplate='<b>%{location}</b><br>Prezzo medio: %{z:,.0f} €<extra></extra>')

In [ ]:
fig_reg = make_subplots(rows=2, cols=3,subplot_titles=["Diesel", "Benzina", "Ibrida", "GPL", "Metano", " "],
                        specs=[[{"type": "choropleth"}] * 3,[{"type": "choropleth"}] * 3], column_widths=[0.33, 0.33, 0.33])
fig_reg.add_trace(get_map_trace(carburante1, geojson, "Diesel", "YlGn"), row=1, col=1)
fig_reg.add_trace(get_map_trace(carburante2, geojson, "Benzina", "YlGn"), row=1, col=2)
fig_reg.add_trace(get_map_trace(carburante3, geojson, "Ibrida", "YlGn"), row=1, col=3)
fig_reg.add_trace(get_map_trace(carburante4, geojson, "GPL", "YlGn"), row=2, col=1)
fig_reg.add_trace(get_map_trace(carburante5, geojson, "Metano", "YlGn"), row=2, col=2)

fig_reg.update_geos(fitbounds="locations", visible=True)
fig_reg.update_layout(title_text="Prezzo Medio per Regione (≤ 40.000 Km)",height=700)

fig_reg.show()

## 3.2. Predizione del prezzo ideale

In [ ]:
data_ml = data_dummy.copy()
data_ml = data_ml[data_ml['Cambio'] != 0]
data_ml = data_ml.drop(columns=['Cambio'])

y = data_ml['Prezzo']
X = data_ml.drop(columns=['Prezzo'])

X_scaled = sc.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X, y)
y_pred = model.predict(X)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_pred)
print(f"Mean Squared Error: {round(mse, 2)}")
print(f"Root Mean Squared Error: {round(rmse, 2)}")
print(f"R-squared:  {round(r2, 2)}")

data_ml['prezzo_previsto'] = y_pred.round(2)
data_ml['delta_prezzo'] = round(data_ml['prezzo_previsto'] - data_ml['Prezzo'],2)
data_ml['is_conveniente'] = (data_ml['delta_prezzo'] > 100).astype(int)

In [ ]:
#@title Distribuzione di is_conveniente
conv_counts = data_ml['is_conveniente'].value_counts()
labels = conv_counts.index
sizes = conv_counts.values

fig_target = px.pie(names=labels,values=sizes,color=labels,color_discrete_map={0: 'lightgreen', 1: 'lightcoral'},title='Distribuzione della variabile dipendente (TARGET)',template='plotly',hole=0.2)

fig_target.update_traces(textinfo='percent+label')
fig_target.update_layout(title ={'text': "Distribuzione della variabile is_conveniente", 'x': 0.5, 'xanchor': 'center','font': {'size': 20, 'family': 'Arial', 'weight': 'bold'}},
                  height=600,width=600,)
fig_target.show()
print(f"\nDistribuzione relativa della variabile BEV dummy: {data_ml['is_conveniente'].value_counts (normalize=True)}" )

In [ ]:
#@title Distribuzione del prezzo predetto
# Parametri per prezzo_previsto
mu_p = data_ml['prezzo_previsto'].mean()
sigma_p = data_ml['prezzo_previsto'].std()
x_p = np.linspace(data_ml['prezzo_previsto'].min(), data_ml['prezzo_previsto'].max(), 300)
y_p = norm.pdf(x_p, mu_p, sigma_p)

# Parametri per delta_prezzo
mu_d = data_ml['delta_prezzo'].mean()
sigma_d = data_ml['delta_prezzo'].std()
x_d = np.linspace(data_ml['delta_prezzo'].min(), data_ml['delta_prezzo'].max(), 300)
y_d = norm.pdf(x_d, mu_d, sigma_d)

# Scaliamo le densità per adattarle all'istogramma
# Altezza istogramma = numero dati * larghezza bin
bin_width_p = (data_ml['prezzo_previsto'].max() - data_ml['prezzo_previsto'].min()) / 120
bin_width_d = (data_ml['delta_prezzo'].max() - data_ml['delta_prezzo'].min()) / 120

y_p_scaled = y_p * len(data_ml['prezzo_previsto']) * bin_width_p
y_d_scaled = y_d * len(data_ml['delta_prezzo']) * bin_width_d

# Creo subplot
fig_dist = make_subplots(rows=1, cols=2,subplot_titles=['Distribuzione di Prezzo Previsto', 'Distribuzione di Delta Prezzo'])

# Istogrammi
fig_dist.add_trace(go.Histogram(x=data_ml['prezzo_previsto'], nbinsx=120, name='Prezzo Previsto', marker_color='steelblue'),row=1, col=1)
fig_dist.add_trace(go.Histogram(x=data_ml['delta_prezzo'], nbinsx=120, name='Delta Prezzo', marker_color='darkorange'),row=1, col=2)

# Curve normali
fig_dist.add_trace(go.Scatter(x=x_p, y=y_p_scaled, mode='lines', name='Curva Normale Prezzo Previsto',line=dict(color='black', dash='dash')),row=1, col=1)

fig_dist.add_trace(go.Scatter(x=x_d, y=y_d_scaled, mode='lines', name='Curva Normale Delta Prezzo',line=dict(color='black', dash='dash')),  row=1, col=2)

fig_dist.update_layout(
    title={'text': "Distribuzione delle variabili Prezzo Previsto e Delta Prezzo",'x': 0.5, 'xanchor': 'center','font': {'size': 20, 'family': 'Arial', 'weight': 'bold'}},
    showlegend=True,bargap=0.05,height=500,width=1300)

fig_dist.show()

In [ ]:
y_class = data_ml['is_conveniente']
X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.2, random_state=42)
model_class = RandomForestClassifier(n_estimators=100, random_state=42)
model_class.fit(X_train, y_train)
y_pred_class = model_class.predict(X_test)
print(classification_report(y_test, y_pred_class))

## 3.3 Indice di Appetibilità

In [ ]:
df = data.copy()

In [ ]:
#Funzioni utili per la creazione dell'indice
def normalizzata_inv(value, min_value, max_value):
    return (max_value - value) / (max_value - min_value)

def normalizzata(value, min_value, max_value):
    return (value - min_value) / (max_value - min_value)


def prezzo_score(x, prezzo_min, prezzo_max):
    soglia = 34000
    if x <= soglia : # Normalizzazione inversa per i prezzi <= soglia
        return (prezzo_max - x) / (prezzo_max - prezzo_min)
    else:
        # Penalizzazione esponenziale severa per i prezzi > 35.000
        base_score = (prezzo_max - soglia) / (prezzo_max - prezzo_min)
        penalty = np.exp(-0.1 * (x - soglia))
        return base_score * penalty

In [ ]:
anni_norm = data['Anni'].apply(lambda x: normalizzata_inv(x, data['Anni'].min(), data['Anni'].max()))
prezzo_norm = data['Prezzo'].apply(lambda x: prezzo_score(x, data['Prezzo'].min(), data['Prezzo'].max()))
chilometraggio_norm = data['Chilometraggio'].apply(lambda x: normalizzata_inv(x, data['Chilometraggio'].min(), data['Chilometraggio'].max()))
distanza_norm = data['Distanza'].apply(lambda x: normalizzata_inv(x, data['Distanza'].min(), data['Distanza'].max()))
cv_norm = data['CV'].apply(lambda x: normalizzata(x, data['CV'].min(), data['CV'].max()))

In [ ]:
# Calcolo dell'indice appetibilità
df['Indice_Appetibilità'] = (
    0.125 * anni_norm +              #Anni bassi è migliore
    0.25 * prezzo_norm +            #Prezzo basso è migliore
    0.20 * chilometraggio_norm +    #Chilometraggio basso è migliore
    0.20 * distanza_norm +          #Distanza bassa è migliore
    0.025 * cv_norm               #CV alto è migliore
)
df['Indice_Appetibilità'] = df['Indice_Appetibilità'].round(2)

In [ ]:
df['prezzo_previsto'] = y_pred.round(2)
df['delta_prezzo'] = round(df['prezzo_previsto'] - df['Prezzo'],2)
df['is_conveniente'] = (df['delta_prezzo'] > 100).astype(int)

# 5. Conclusione

Le auto migliori secondo l'Indice di Appettibilità sono:

In [ ]:
#@title Le 10 auto migliori con prezzo > 12.000€
over_30k = df[df['Prezzo'] > 12000]
top_10_over = over_30k.nlargest(10, 'Indice_Appetibilità')
for index, row in top_10_over.iterrows():
    link = row['Link']
    display(HTML(f"Indice {row['Indice_Appetibilità']:.2f} → <a href='{link}' target='_blank'>{link}</a>"))
print()
display(top_10_over)

In [ ]:
#@title Le 10 auto migliori con prezzo dai 8 ai 12.000€
under_30k = df[(df['Prezzo'] <= 12000) & (df['Prezzo'] > 8000)]
top_10_middle = under_30k.nlargest(10, 'Indice_Appetibilità')
for index, row in top_10_middle.iterrows():
    link = row['Link']
    display(HTML(f"Indice {row['Indice_Appetibilità']:.2f} → <a href='{link}' target='_blank'>{link}</a>"))
print()
display(top_10_middle)

In [ ]:
#@title Le 10 auto migliori con prezzo ≤ 8.000€
under_25k = df[df['Prezzo'] <= 8000]
top_10_under = under_25k.nlargest(10, 'Indice_Appetibilità')
for index, row in top_10_under.iterrows():
    link = row['Link']
    display(HTML(f"Indice {row['Indice_Appetibilità']:.2f} → <a href='{link}' target='_blank'>{link}</a>"))
print()
display(top_10_under)

In [ ]:
#@title Le 10 auto migliori secondo l'Indice di Appettibilità
top_ten = df.nlargest(10, 'Indice_Appetibilità')
for index, row in top_ten.iterrows():
    link = row['Link']
    display(HTML(f"Numero indice: {index} -> <a href='{link}' target='_blank'>{link}</a>"))
print()
top_ten